# as-strided-windowing — worked example 2: Dilated 1-D windows (gaps between window elements) via as_strided

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `as-strided-windowing`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A *dilated* window of width `K` and dilation `d` covers indices `i, i+d, i+2d, ..., i+(K-1)d` instead of `K` adjacent elements. With `as_strided`, dilation multiplies only the **inner** stride: `stride=(s, s*d)`. The outer stride stays `s` so consecutive windows still start one element apart. This is the building block behind dilated/atrous convolutions.

## Worked solution

**Goal.** Produce a `(L_out, K)` view where row `i` is `x[i], x[i+d], x[i+2d], ..., x[i+(K-1)d]` — a dilated sliding window.

1. **Source stride.** `s, = x.stride()`. As always, never hard-code `1`.
2. **Effective span of one window.** A dilated window of `K` taps with dilation `d` physically spans `(K-1)*d + 1` elements. So the last valid start index is `L - ((K-1)*d + 1)`, giving `L_out = L - (K-1)*d - 1 + 1 = L - (K-1)*d`. Equivalently `L_out = (L - ((K-1)*d + 1)) // 1 + 1`.
3. **Strides.** The inner axis must jump `d` logical elements per tap, i.e. `s*d` storage units. The outer axis advances the window origin by one element, i.e. `s`. Hence `stride=(s, s*d)`.
4. **Build the view.** `t.as_strided(x, size=(L_out, K), stride=(s, s*d))`. Still a zero-copy alias of `x`.

**Why it works.** `as_strided` lets the two axes use independent strides. Decoupling the *step between windows* (outer = `s`) from the *step within a window* (inner = `s*d`) is precisely what dilation needs. Setting `d=1` recovers the ordinary contiguous window.

In [ ]:
def dilated_1d_windows(x: Tensor, K: int, d: int) -> Tensor:
    L = x.shape[0]
    s, = x.stride()
    L_out = L - (K - 1) * d
    return t.as_strided(x, size=(L_out, K), stride=(s, s * d))

t.manual_seed(0)
x = t.arange(10, dtype=t.float32)        # [0..9]
w = dilated_1d_windows(x, K=3, d=2)      # row i = [i, i+2, i+4]
print(w.shape)                           # torch.Size([6, 3])
print(w[0])                              # [0, 2, 4]
print(w[-1])                             # [5, 7, 9]